Когорты сформированы по месяцу первой покупки.
Retention на n-ый день считается как доля клиентов когорты, совершивших повторную покупку в течение первых n дней 
(1 ≤ days_since_first_purchase ≤ n)

In [2]:
import pandas as pd
import numpy as np

In [6]:
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)

tx = pd.read_csv("raw_files/transactions_with_cohorts.csv", parse_dates = ["t_dat", "first_purchase_date"])
tx.head()

,customer_id,t_dat,first_purchase_date,cohort_month,days_since_first_purchase
0,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2018-09-20,2018-09-20,2018-09,0
1,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,2018-09-20,2018-09-20,2018-09,0
2,00007d2de826758b65a93dd24ce629ed66842531df6699...,2018-09-20,2018-09-20,2018-09,0
3,00007d2de826758b65a93dd24ce629ed66842531df6699...,2018-09-20,2018-09-20,2018-09,0
4,00007d2de826758b65a93dd24ce629ed66842531df6699...,2018-09-20,2018-09-20,2018-09,0


In [7]:
print("Rows:", len(tx))
print("Unique customers:", tx["customer_id"].nunique())
print("Cohorts:", tx["cohort_month"].nunique())
print("Date range:", tx["t_dat"].min().date(), "-", tx["t_dat"].max().date())

Rows: 31788324
Unique customers: 1362281
Cohorts: 25
Date range: 2018-09-20 - 2020-09-22


In [8]:
cohort_sizes = (
    tx.groupby("cohort_month")["customer_id"]
      .nunique()
      .rename("cohort_size")
      .reset_index()
      .sort_values("cohort_month")
)
cohort_sizes.head()

,cohort_month,cohort_size
0,2018-09,140340
1,2018-10,212669
2,2018-11,138233
3,2018-12,89944
4,2019-01,68957


Будем считать клиента удержанным на день n, если у него есть хотя бы одна повторная покупка в окне от 1 до n дней.
Исключаем день 0, чтобы не засчитывать первую покупку.

In [ ]:
def retention_cumulative(df: pd.DataFrame, days: list[int]) -> pd.DataFrame:
    out = cohort_sizes.copy()
    for d in days:
        retained = (
            df[(df["days_since_first_purchase"] >= 1) & (df["days_since_first_purchase"] <= d)]
            .groupby("cohort_month")["customer_id"].nunique()
            .rename(f"retained_upto_d{d}")
            .reset_index()
        )
        out = out.merge(retained, on = "cohort_month", how = "left")
        out[f"retained_upto_d{d}"] = out[f"retained_upto_d{d}"].fillna(0).astype(int)
        out[f"retention_upto_d{d}"] = out[f"retained_upto_d{d}"] / out["cohort_size"]
    return out

In [11]:
days_list = [7, 30, 90]
ret_cum = retention_cumulative(tx, days_list)
ret_cum.head()

,cohort_month,cohort_size,retained_upto_d7,retention_upto_d7,retained_upto_d30,retention_upto_d30,retained_upto_d90,retention_upto_d90
0,2018-09,140340,26229,0.187,63191,0.450,97553,0.695
1,2018-10,212669,32496,0.153,80975,0.381,133579,0.628
2,2018-11,138233,17698,0.128,43179,0.312,72018,0.521
3,2018-12,89944,10848,0.121,23968,0.266,41237,0.458
4,2019-01,68957,7618,0.110,16402,0.238,29581,0.429


Для расчёта удержания на 90-й день учитываются только те когорты, для которых доступен полный период наблюдения. Для остальных когорт значение d90 помечается как NaN, а не интерпретируется как нулевое удержание.

In [12]:
def eligible_cohorts(df: pd.DataFrame, horizon_days: int) -> set:
    max_date = df["t_dat"].max()
    cohort_start = (
        df.groupby("cohort_month")["first_purchase_date"]
          .min()
    )
    eligible = cohort_start[cohort_start <= (max_date - pd.Timedelta(days = horizon_days))].index
    return set(eligible)

In [14]:
eligible_90 = eligible_cohorts(tx, 90)
print("Eligible for d90:", len(eligible_90), "/", tx["cohort_month"].nunique())

Eligible for d90: 22 / 25


In [15]:
ret_cum_adj = ret_cum.copy()
mask_90 = ret_cum_adj["cohort_month"].isin(eligible_90)
ret_cum_adj.loc[~mask_90, ["retention_upto_d90", "retained_upto_d90"]] = np.nan
ret_cum_adj.tail()

,cohort_month,cohort_size,retained_upto_d7,retention_upto_d7,retained_upto_d30,retention_upto_d30,retained_upto_d90,retention_upto_d90
20,2020-05,30518,3701,0.121,7238,0.237,11090.000,0.363
21,2020-06,31685,4023,0.127,7385,0.233,10866.000,0.343
22,2020-07,23563,2539,0.108,4796,0.204,NaN,NaN
23,2020-08,22974,2260,0.098,4151,0.181,NaN,NaN
24,2020-09,17538,1402,0.080,1806,0.103,NaN,NaN


In [16]:
df_retention = ret_cum_adj[["cohort_month", "retention_upto_d7", "retention_upto_d30", "retention_upto_d90"]].copy()
df_retention = df_retention.sort_values("cohort_month")
df_retention.head()

,cohort_month,retention_upto_d7,retention_upto_d30,retention_upto_d90
0,2018-09,0.187,0.450,0.695
1,2018-10,0.153,0.381,0.628
2,2018-11,0.128,0.312,0.521
3,2018-12,0.121,0.266,0.458
4,2019-01,0.110,0.238,0.429


Проверим, что удержание возрастающее с увеличением дня.

In [21]:
tmp = ret_cum_adj.dropna(subset = ["retention_upto_d90"])
print(((tmp["retention_upto_d7"] <= tmp["retention_upto_d30"]) & (tmp["retention_upto_d30"] <= tmp["retention_upto_d90"])).mean())

1.0


In [22]:
df_retention[["retention_upto_d7","retention_upto_d30","retention_upto_d90"]].describe()

,retention_upto_d7,retention_upto_d30,retention_upto_d90
count,25.000,25.000,22.000
mean,0.110,0.230,0.398
std,0.025,0.070,0.104
min,0.063,0.103,0.298
25%,0.095,0.186,0.326
50%,0.110,0.218,0.361
75%,0.121,0.243,0.432
max,0.187,0.450,0.695


In [20]:
df_retention.to_csv("processed_files/retention_table.csv", index=False)